# Predict: Solar Filament Segmentation
Load trained model, run inference on the test set, write `submission.csv`.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("segmentation_models_pytorch") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"],
        check=True,
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.5 MB/s eta 0:00:00


In [2]:
import os

import numpy as np
import pandas as pd
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from pycocotools import mask as mask_utils
from scipy import ndimage
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

In [3]:
INPUT_ROOT = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
OUTPUT_ROOT = "/kaggle/working"

CONFIG = {
    "test_images_dir": f"{INPUT_ROOT}/test/test_images",
    "checkpoint_model": f"/kaggle/input/models/kumail420/segres-epochs-1-5/pytorch/default/3/model.pt",
    "submission_path": f"{OUTPUT_ROOT}/submission.csv",
    "image_height": 2048,
    "image_width": 2048,
    "tile_size": 512,
    "overlap": 64,
    "batch_size": 8,
    "num_workers": 2,
    "pred_threshold": 0.97,
    "min_area": 100,
}

#### Tiling

In [4]:
def _axis_coordinates(size, tile_size, stride):
    """
    Top-left coordinates along one axis, evenly spaced to exactly cover
    [0, size) with `tile_size`-wide tiles.
    """
    if size <= tile_size:
        return [0]

    n_tiles = int(np.ceil((size - tile_size) / stride)) + 1
    positions = np.linspace(0, size - tile_size, n_tiles)
    return sorted({int(round(p)) for p in positions})


def get_tile_coordinates(height, width, tile_size=512, overlap=64):
    """Top-left (y, x) coordinates for tiles covering the full image, with overlap."""
    stride = tile_size - overlap
    ys = _axis_coordinates(height, tile_size, stride)
    xs = _axis_coordinates(width, tile_size, stride)
    return [(y, x) for y in ys for x in xs]


def extract_tile(array, y, x, tile_size=512):
    """Extract a single tile from a 2D array (image or mask)."""
    tile = array[y:y + tile_size, x:x + tile_size]
    return np.ascontiguousarray(tile)


def stitch_predictions(pred_tiles, coords, height, width, tile_size=512):
    """Reassemble predicted tiles into a full-resolution mask, averaging overlaps."""
    full_pred = np.zeros((height, width), dtype=np.float32)
    count_map = np.zeros((height, width), dtype=np.float32)

    for pred_tile, (y, x) in zip(pred_tiles, coords):
        full_pred[y:y + tile_size, x:x + tile_size] += pred_tile
        count_map[y:y + tile_size, x:x + tile_size] += 1.0

    count_map[count_map == 0] = 1.0
    full_pred = full_pred / count_map

    return full_pred

#### Dataset

In [5]:
class SolarFilamentTestDataset(Dataset):
    def __init__(self, file_names, images_dir, transform=None, tile_size=512,
                 overlap=64, image_height=2048, image_width=2048):
        """
        Test-time dataset: tiles a directory of full images, no annotations/masks.
        transform: an Albumentations transform, applied to the image tile only.
        """
        self.images_dir = images_dir
        self.transform = transform
        self.tile_size = tile_size

        self.tile_coords = get_tile_coordinates(image_height, image_width, tile_size, overlap)

        self.index = []
        for file_name in file_names:
            for (y, x) in self.tile_coords:
                self.index.append((file_name, y, x))

        self._images = {}

    def __len__(self):
        return len(self.index)

    def _load_image(self, file_name):
        if file_name not in self._images:
            image_path = os.path.join(self.images_dir, file_name)
            image = Image.open(image_path).convert("L")  # force grayscale
            self._images[file_name] = np.array(image)
        return self._images[file_name]

    def __getitem__(self, idx):
        file_name, y, x = self.index[idx]

        image = self._load_image(file_name)
        image_tile = extract_tile(image, y, x, self.tile_size)

        if self.transform:
            image_tile = self.transform(image=image_tile)["image"]

        # y, x returned so predictions can be stitched back later
        return image_tile, file_name, y, x


def list_image_files(images_dir):
    return sorted(f for f in os.listdir(images_dir) if f.lower().endswith((".jpeg", ".jpg", ".png")))

#### Transforms

In [6]:
# Single-channel, but the resnet34 encoder is ImageNet-pretrained, and smp
# adapts it to in_channels=1 by summing the RGB conv weights, so it still
# expects ImageNet-normalized input. These are the ImageNet RGB stats
# collapsed to luminance (0.299R + 0.587G + 0.114B).
GRAYSCALE_MEAN = (0.449,)
GRAYSCALE_STD = (0.226,)


def get_test_transform():
    return A.Compose([
        A.Normalize(mean=GRAYSCALE_MEAN, std=GRAYSCALE_STD),
        ToTensorV2(),
    ])

#### Model

In [7]:
def build_model(encoder_weights=None):
    """
    encoder_weights=None: the checkpoint overwrites these weights anyway, and
    the imagenet download hard-fails in a no-internet Kaggle inference kernel.
    """
    return smp.UnetPlusPlus(
        encoder_name="resnet34",
        encoder_weights=encoder_weights,
        in_channels=1,
        classes=1,
    )


def load_model(checkpoint_path, device):
    model = build_model()
    state = torch.load(checkpoint_path, map_location=device)
    # training checkpoints wrap weights under "model"; inference-only ones don't
    state_dict = state["model"] if isinstance(state, dict) and "model" in state else state
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

#### Predict

In [8]:
def predict_tiles(model, loader, device):
    """
    Run the model on every test tile, grouped by file_name for stitching.
    Returns: dict {file_name: {"tiles": [...], "coords": [...]}}
    """
    results = {}

    with torch.no_grad():
        for images, file_names, ys, xs in loader:
            images = images.to(device)
            preds = torch.sigmoid(model(images)).cpu().numpy()  # (B, 1, H, W)

            for i in range(len(file_names)):
                file_name = file_names[i]
                y, x = ys[i].item(), xs[i].item()
                pred_tile = preds[i, 0]

                entry = results.setdefault(file_name, {"tiles": [], "coords": []})
                entry["tiles"].append(pred_tile)
                entry["coords"].append((y, x))

    return results


def label_and_filter(binary_mask, min_area=1):
    """
    Label connected components and drop tiny specks by area.
    Returns: (cleaned_binary_mask, [instance_mask, ...])
    """
    labeled, num_features = ndimage.label(binary_mask)
    if num_features == 0:
        return np.zeros_like(binary_mask, dtype=np.uint8), []

    areas = np.bincount(labeled.ravel(), minlength=num_features + 1)
    keep_ids = np.nonzero(areas[1:] >= min_area)[0] + 1

    cleaned = np.isin(labeled, keep_ids).astype(np.uint8)
    instances = [(labeled == label_id).astype(np.uint8) for label_id in keep_ids]

    return cleaned, instances


def mask_to_rle_string(binary_mask):
    """Convert a binary mask to an RLE counts string, per the submission format."""
    rle = mask_utils.encode(np.asfortranarray(binary_mask))
    counts = rle["counts"]
    if isinstance(counts, bytes):
        counts = counts.decode("utf-8")
    return counts


def build_submission(results, threshold, min_area, output_csv, tile_size, image_height, image_width):
    rows = []

    for file_name, data in results.items():
        stitched = stitch_predictions(data["tiles"], data["coords"], image_height, image_width, tile_size)
        binary_mask = (stitched > threshold).astype(np.uint8)
        _, instances = label_and_filter(binary_mask, min_area=min_area)

        base_name = os.path.splitext(file_name)[0]
        for idx, instance_mask in enumerate(instances, start=1):
            rows.append({
                "filament_id": f"{base_name}_{idx}",
                "segmentation_rle": mask_to_rle_string(instance_mask),
            })

        # every test image needs at least one row, even with no surviving filament
        if not instances:
            empty_mask = np.zeros((image_height, image_width), dtype=np.uint8)
            rows.append({
                "filament_id": f"{base_name}_1",
                "segmentation_rle": mask_to_rle_string(empty_mask),
            })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"Saved submission with {len(df)} rows to {output_csv}")

#### Run

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

using device: cuda


In [10]:
test_files = list_image_files(CONFIG["test_images_dir"])

test_dataset = SolarFilamentTestDataset(
    test_files,
    images_dir=CONFIG["test_images_dir"],
    transform=get_test_transform(),
    tile_size=CONFIG["tile_size"],
    overlap=CONFIG["overlap"],
    image_height=CONFIG["image_height"],
    image_width=CONFIG["image_width"],
)
test_loader = DataLoader(
    test_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=True,
)

model = load_model(CONFIG["checkpoint_model"], device)
results = predict_tiles(model, test_loader, device)

os.makedirs(os.path.dirname(CONFIG["submission_path"]), exist_ok=True)
build_submission(
    results,
    threshold=CONFIG["pred_threshold"],
    min_area=CONFIG["min_area"],
    output_csv=CONFIG["submission_path"],
    tile_size=CONFIG["tile_size"],
    image_height=CONFIG["image_height"],
    image_width=CONFIG["image_width"],
)

Saved submission with 2822 rows to /kaggle/working/submission.csv
